Contour Properties 
==================

@prev_tutorial{tutorial_py_contour_features}
@next_tutorial{tutorial_py_contours_more_functions}

Here we will learn to extract some frequently used properties of objects like Solidity, Equivalent
Diameter, Mask image, Mean Intensity etc. More features can be found at [Matlab regionprops
documentation](http://www.mathworks.in/help/images/ref/regionprops.html).

*(NB : Centroid, Area, Perimeter etc also belong to this category, but we have seen it in last
chapter)*

1. Aspect Ratio
---------------

It is the ratio of width to height of bounding rect of the object.

$$Aspect \; Ratio = \frac{Width}{Height}$$

In [ ]:
import cv2 as cv

x, y, w, h = cv.boundingRect(cnt)
aspect_ratio = float(w) / h

2. Extent
---------

Extent is the ratio of contour area to bounding rectangle area.

$$Extent = \frac{Object \; Area}{Bounding \; Rectangle \; Area}$$

In [ ]:
area = cv.contourArea(cnt)
x, y, w, h = cv.boundingRect(cnt)
rect_area = w * h
extent = float(area) / rect_area

3. Solidity
-----------

Solidity is the ratio of contour area to its convex hull area.

$$Solidity = \frac{Contour \; Area}{Convex \; Hull \; Area}$$

In [ ]:
area = cv.contourArea(cnt)
hull = cv.convexHull(cnt)
hull_area = cv.contourArea(hull)
solidity = float(area) / hull_area

4. Equivalent Diameter
----------------------

Equivalent Diameter is the diameter of the circle whose area is same as the contour area.

$$Equivalent \; Diameter = \sqrt{\frac{4 \times Contour \; Area}{\pi}}$$

In [ ]:
area = cv.contourArea(cnt)
equi_diameter = np.sqrt(4 * area / np.pi)

5. Orientation
--------------

Orientation is the angle at which object is directed. Following method also gives the Major Axis and
Minor Axis lengths.

In [ ]:
(x, y), (MA, ma), angle = cv.fitEllipse(cnt)

6. Mask and Pixel Points
------------------------

In some cases, we may need all the points which comprises that object. It can be done as follows:

In [ ]:
mask = np.zeros(imgray.shape, np.uint8)
cv.drawContours(mask, [cnt], 0, 255, -1)
pixelpoints = np.transpose(np.nonzero(mask))
# pixelpoints = cv.findNonZero(mask)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(cv.cvtColor(img1, cv.COLOR_BGR2RGB))

Here, two methods, one using Numpy functions, next one using OpenCV function (last commented line)
are given to do the same. Results are also same, but with a slight difference. Numpy gives
coordinates in **(row, column)** format, while OpenCV gives coordinates in **(x,y)** format. So
basically the answers will be interchanged. Note that, **row = y** and **column = x**.

7. Maximum Value, Minimum Value and their locations
---------------------------------------------------

We can find these parameters using a mask image.

In [ ]:
min_val, max_val, min_loc, max_loc = cv.minMaxLoc(imgray, mask=mask)

8. Mean Color or Mean Intensity
-------------------------------

Here, we can find the average color of an object. Or it can be average intensity of the object in
grayscale mode. We again use the same mask to do it.

In [ ]:
mean_val = cv.mean(im, mask=mask)

9. Extreme Points
-----------------

Extreme Points means topmost, bottommost, rightmost and leftmost points of the object.

In [ ]:
leftmost = tuple(cnt[cnt[:, :, 0].argmin()][0])
rightmost = tuple(cnt[cnt[:, :, 0].argmax()][0])
topmost = tuple(cnt[cnt[:, :, 1].argmin()][0])
bottommost = tuple(cnt[cnt[:, :, 1].argmax()][0])

For eg, if I apply it to an Indian map, I get the following result :

![image](images/extremepoints.jpg)

Exercises
---------

1. There are still some features left in matlab regionprops doc. Try to implement them.

---

## 🎛️ Interactive Exploration — Contour Explorer

The cells above reproduce the original tutorial.  Below is an interactive version: adjust the widgets and the ipycanvas Canvas refreshes live.


In [ ]:
import ipywidgets as widgets
from ipycanvas import MultiCanvas, hold_canvas

from common import to_rgb

src = cv.imread(f"{ASSETS}/messi5.jpg")
src = cv.resize(src, (360, 240))
gray = cv.cvtColor(src, cv.COLOR_BGR2GRAY)

mc = MultiCanvas(2, width=src.shape[1], height=src.shape[0])
bg, fg = mc[0], mc[1]
bg.put_image_data(to_rgb(src), 0, 0)

th_sl = widgets.IntSlider(value=100, min=0, max=255, description="Canny thresh")
mode_dd = widgets.Dropdown(
    options=["External", "List", "Tree", "CComp"], value="External", description="Mode"
)
prop_dd = widgets.Dropdown(
    options=["None", "Bounding Box", "Convex Hull", "Approx Polygon"],
    value="None",
    description="Draw",
)
modes = {
    "External": cv.RETR_EXTERNAL,
    "List": cv.RETR_LIST,
    "Tree": cv.RETR_TREE,
    "CComp": cv.RETR_CCOMP,
}


def update(*_):
    edges = cv.Canny(gray, th_sl.value, th_sl.value * 2)
    cnts, _ = cv.findContours(edges, modes[mode_dd.value], cv.CHAIN_APPROX_SIMPLE)
    with hold_canvas():
        fg.clear()
        fg.stroke_style = "lime"
        fg.line_width = 2
        for c in cnts:
            if prop_dd.value == "None":
                pts = [tuple(p[0]) for p in c]
                fg.stroke_lines(pts)
            elif prop_dd.value == "Bounding Box":
                x, y, w, h = cv.boundingRect(c)
                fg.stroke_rect(x, y, w, h)
            elif prop_dd.value == "Convex Hull":
                hull = cv.convexHull(c)
                pts = [tuple(p[0]) for p in hull]
                fg.stroke_lines(pts)
            elif prop_dd.value == "Approx Polygon":
                eps = 0.02 * cv.arcLength(c, True)
                approx = cv.approxPolyDP(c, eps, True)
                pts = [tuple(p[0]) for p in approx]
                fg.stroke_lines(pts)


for w in (th_sl, mode_dd, prop_dd):
    w.observe(update, "value")
update()

display(widgets.VBox([widgets.HBox([th_sl, mode_dd, prop_dd]), mc]))